## Import Libraries

In [ ]:
from youtube_transcript_api import YouTubeTranscriptApi, TranscriptsDisabled
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate
import pprint

### 1. Indexing

In [ ]:
YouTubeTranscriptApi.fetch()

In [ ]:
video_id = 'bzEO6RngvL4' #Chris Bumstead video
try:
    transcript_list = YouTubeTranscriptApi().fetch(video_id=video_id, languages=['en'])
    transcript = " ".join(
                        snippet.text for snippet in transcript_list
                    )
    print(transcript)
except TranscriptsDisabled:
    print("No Captions available for this video")

#### 1b. Text Splitting

In [ ]:
splitter = RecursiveCharacterTextSplitter(chunk_size = 800, chunk_overlap = 200)
chunks = splitter.create_documents([transcript])

#### 1c. Embedding Generation and storing in vector store

In [ ]:
embeddings = OpenAIEmbeddings(model= 'text-embedding-3-small')
vector_store= FAISS.from_documents(chunks, embeddings)

### 2. Retrieval

In [ ]:
retriever = vector_store.as_retriever(search_type= 'similarity', search_kwargs= {'k':4})

### 3. Augmentation

In [ ]:
llm = ChatOpenAI(model= 'gpt-6-luna', reasoning_effort="medium")

In [ ]:
prompt = PromptTemplate(
    template="""
    You are a helpful assistant, who analyses the question deeply and give the relevant answer only from the transcript.
    If the context is insufficient just say "Humpe to hai hi no".

    {context}
    Question: {question}
    """,
    input_variables=['context','question']
)

In [ ]:
question = "Summarise the video"
retrieved_docs = retriever.invoke(question)

In [ ]:
context_text= "\n\n".join(docs.page_content for docs in retrieved_docs)

In [ ]:
final_prompt = prompt.invoke({'context':context_text, 'question':question})

### 4. Generation

In [ ]:
answer = llm.invoke(final_prompt)

### 5. Building Chain

In [ ]:
from langchain_core.runnables import RunnableParallel, RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser

In [ ]:
def format_docs(retrieved_docs):
    context_text= "\n\n".join(doc.page_content for doc in retrieved_docs)
    return context_text

In [ ]:
parallel_chain = RunnableParallel({
    'context': retriever | RunnableLambda(format_docs),
    'question': RunnablePassthrough()
})

In [ ]:
parser= StrOutputParser()

In [ ]:
main_chain = parallel_chain | prompt | llm | parser

In [ ]:
main_chain.invoke('What is discipline?')